In [ ]:
import numpy as np
import openpyxl
from datetime import datetime
import calendar
import sys

DATA_DIR = "/workspace/data/"
wb = openpyxl.load_workbook(DATA_DIR + "MO13-Hard-Times-Assumptions.xlsx", data_only=True)
ws = wb[wb.sheetnames[0]]

# Read all data
raw = []
for row in ws.iter_rows(min_row=1, max_row=ws.max_row, max_col=ws.max_column, values_only=True):
    raw.append(list(row))

# Print for debugging
for r_idx, row in enumerate(raw):
    non_none = [(c, v) for c, v in enumerate(row) if v is not None]
    if non_none:
        print(f"Row {r_idx}: {non_none}", file=sys.stderr)

In [ ]:
# ============================================================
# EXTRACT ALL PARAMETERS FROM EXCEL
# ============================================================

def find_label(data, label):
    for r, row in enumerate(data):
        for c, val in enumerate(row):
            if val is not None and isinstance(val, str) and label.lower() in val.lower():
                return r, c
    return None, None

def nums_after(data, r, c):
    if r is None: return []
    return [v for j, v in enumerate(data[r]) if j > c and isinstance(v, (int, float)) and not isinstance(v, bool)]

# Seasonality
season_q = [0.20, 0.15, 0.30, 0.35]
r, c = find_label(raw, "quarterly seasonality")
if r is not None:
    for check in range(r, min(r+5, len(raw))):
        vals = [v for v in raw[check] if isinstance(v, (int, float)) and not isinstance(v, bool)]
        if len(vals) == 4 and 0.95 < sum(vals) < 1.05:
            season_q = vals
            break
print(f"Seasonality: {season_q}")

# Units sold
r, c = find_label(raw, "units sold")
base_units = nums_after(raw, r, c)[0] if r else 175000
print(f"Base units 2013: {base_units:,.0f}")

# Sales growth per year (different each year: 4%, 12%, 12%)
r, c = find_label(raw, "sales growth")
growth_vals = nums_after(raw, r, c) if r else [0.04, 0.12, 0.12]
print(f"Sales growth: {growth_vals}")

# Compute annual units with per-year growth rates
cy_units = {2013: base_units}
for i, yr in enumerate([2014, 2015, 2016]):
    g = growth_vals[i] if i < len(growth_vals) else growth_vals[-1]
    cy_units[yr] = cy_units[yr-1] * (1 + g)
print(f"Annual units: {dict((k, f'{v:,.0f}') for k, v in cy_units.items())}")

# Sale price
r, c = find_label(raw, "sale price")
sale_price = nums_after(raw, r, c)[0] if r else 29.99
print(f"Sale price: ${sale_price}")

# Contribution margin (ratio)
r, c = find_label(raw, "contribution margin")
cm_ratio = nums_after(raw, r, c)[0] if r else 0.25
if cm_ratio > 1: cm_ratio /= 100
dc_per_unit = sale_price * (1 - cm_ratio)
cm_per_unit = sale_price * cm_ratio
print(f"CM ratio: {cm_ratio}, DC/unit: ${dc_per_unit:.4f}, CM/unit: ${cm_per_unit:.4f}")

# Indirect costs
r, c = find_label(raw, "indirect costs")
ic_base = None
if r is not None:
    for check in range(r, min(r+5, len(raw))):
        vals = nums_after(raw, check, c)
        big_vals = [v for v in vals if v > 10000]
        if big_vals:
            ic_base = big_vals[0]
            break
if ic_base is None: ic_base = 1200000
print(f"IC base 2013: ${ic_base:,.0f}")

# IC growth
r, c = find_label(raw, "indirect cost growth")
icg_vals = nums_after(raw, r, c) if r else [0.04]
ic_growth = icg_vals[0] if icg_vals else 0.04
if ic_growth > 1: ic_growth /= 100
print(f"IC growth: {ic_growth}")

cy_indirect = {}
for yr in [2013, 2014, 2015, 2016]:
    cy_indirect[yr] = ic_base * (1 + ic_growth) ** (yr - 2013)
print(f"Annual IC: {dict((k, f'${v:,.0f}') for k, v in cy_indirect.items())}")

# Cash receipts on sales: 0%, 60%, 25%, 15%
r, c = find_label(raw, "cash receipts on sales")
cr_timing = [v for v in (nums_after(raw, r, c) if r else [])][:4]
if not cr_timing: cr_timing = [0, 0.60, 0.25, 0.15]
print(f"CR on sales: {cr_timing}")

# Cash receipts on opening AR: 30%, 60%, 10%
r, c = find_label(raw, "cash receipts on opening")
cr_open = [v for v in (nums_after(raw, r, c) if r else [])][:3]
if not cr_open: cr_open = [0.30, 0.60, 0.10]
print(f"CR on opening AR: {cr_open}")

# Cash payments on purchases: 0%, 85%, 10%, 5%
r, c = find_label(raw, "cash payments on purchases")
cp_timing = [v for v in (nums_after(raw, r, c) if r else [])][:4]
if not cp_timing: cp_timing = [0, 0.85, 0.10, 0.05]
print(f"CP on purchases: {cp_timing}")

# Cash payments on opening AP: 30%, 60%, 10%
r, c = find_label(raw, "cash payments on opening")
cp_open = [v for v in (nums_after(raw, r, c) if r else [])][:3]
if not cp_open: cp_open = [0.30, 0.60, 0.10]
print(f"CP on opening AP: {cp_open}")

# Balance sheet opening values
open_cash = 18000
open_ar = 600000
open_ppe = 500000
open_ap = 350000
open_debt = 2000000
interest_rate = 0.07
depreciation_monthly = 10000

for row_idx in range(len(raw)):
    for col_idx, val in enumerate(raw[row_idx]):
        if val is not None and isinstance(val, str):
            vl = val.strip().lower()
            row_nums = [v for j, v in enumerate(raw[row_idx]) if j > col_idx
                       and isinstance(v, (int, float)) and not isinstance(v, bool)]
            if vl == 'cash' and row_nums:
                open_cash = row_nums[0]
            elif 'accounts receivable' in vl and row_nums:
                open_ar = row_nums[0]
            elif 'property' in vl and 'plant' in vl and row_nums:
                open_ppe = row_nums[0]
            elif 'accounts payable' in vl and row_nums:
                open_ap = row_nums[0]
            elif 'debt facility' in vl and row_nums:
                open_debt = row_nums[0]
                if len(row_nums) > 1 and row_nums[1] < 1:
                    interest_rate = row_nums[1]
            elif vl == 'depreciation' and row_nums:
                depreciation_monthly = row_nums[0]

print(f"\nOpening BS: Cash=${open_cash:,.0f}, AR=${open_ar:,.0f}, PPE=${open_ppe:,.0f}")
print(f"  AP=${open_ap:,.0f}, Debt=${open_debt:,.0f}, Rate={interest_rate}")
print(f"  Depreciation: ${depreciation_monthly:,.0f}/month")

# Debt amortization schedule - scan for all date+amount pairs
debt_schedule = {}
r_amort, _ = find_label(raw, "amortisation")
if r_amort is None:
    r_amort, _ = find_label(raw, "amortization")
if r_amort is not None:
    for check in range(r_amort+1, min(r_amort+20, len(raw))):
        row_data = raw[check]
        dates = [v for v in row_data if isinstance(v, datetime)]
        amounts = [v for v in row_data if isinstance(v, (int, float)) and not isinstance(v, bool) and v > 0]
        if dates and amounts:
            d = dates[0]
            debt_schedule[(d.year, d.month)] = amounts[0]
        # Don't break on empty/header rows - keep scanning

print(f"\nDebt amortization schedule ({len(debt_schedule)} entries):")
for (y, m), a in sorted(debt_schedule.items()):
    print(f"  {y}-{m:02d}: ${a:,.0f}")
print(f"  Total: ${sum(debt_schedule.values()):,.0f}")

In [ ]:
# ============================================================
# BUILD 36-MONTH FORECAST MODEL (Oct 2013 - Sep 2016)
# ============================================================

months = []
for i in range(36):
    year = 2013 + (9 + i) // 12
    month = ((9 + i) % 12) + 1
    months.append((year, month))

N = len(months)

def month_idx(year, month):
    for i, (y, m) in enumerate(months):
        if y == year and m == month:
            return i
    return None

# Monthly units: equal within quarter (per instruction)
monthly_units = []
for yr, mo in months:
    q = (mo - 1) // 3  # 0=Q1(Jan-Mar), 1=Q2(Apr-Jun), 2=Q3(Jul-Sep), 3=Q4(Oct-Dec)
    annual = cy_units[yr]
    monthly_units.append(annual * season_q[q] / 3)

# Revenue and costs
monthly_revenue = [u * sale_price for u in monthly_units]
monthly_dc = [u * dc_per_unit for u in monthly_units]
monthly_ic = [cy_indirect[yr] / 12 for yr, mo in months]
monthly_total_cost = [d + ic for d, ic in zip(monthly_dc, monthly_ic)]
monthly_gp = [r - d for r, d in zip(monthly_revenue, monthly_dc)]
monthly_ebit = [gp - ic - depreciation_monthly for gp, ic in zip(monthly_gp, monthly_ic)]

# Debt schedule with specific amortization dates
debt_bal = [0.0] * (N + 1)
debt_bal[0] = open_debt
m_interest = [0.0] * N
m_debt_repay = [0.0] * N

for i in range(N):
    yr, mo = months[i]
    m_interest[i] = debt_bal[i] * interest_rate / 12
    repay = debt_schedule.get((yr, mo), 0)
    repay = min(repay, debt_bal[i])  # can't repay more than balance
    m_debt_repay[i] = repay
    debt_bal[i + 1] = debt_bal[i] - repay

# PPE schedule
ppe_bal = [0.0] * (N + 1)
ppe_bal[0] = open_ppe
for i in range(N):
    ppe_bal[i + 1] = ppe_bal[i] - depreciation_monthly

# Cash receipts
cash_in = [0.0] * N
# Opening AR collections (Oct, Nov, Dec 2013)
for j, pct in enumerate(cr_open):
    if j < N:
        cash_in[j] += open_ar * pct
# Sales collections
for i in range(N):
    for j, pct in enumerate(cr_timing):
        t = i + j
        if t < N:
            cash_in[t] += monthly_revenue[i] * pct

# Cash payments
cash_out = [0.0] * N
# Opening AP payments (Oct, Nov, Dec 2013)
for j, pct in enumerate(cp_open):
    if j < N:
        cash_out[j] += open_ap * pct
# Cost payments (both direct and indirect)
for i in range(N):
    for j, pct in enumerate(cp_timing):
        t = i + j
        if t < N:
            cash_out[t] += monthly_total_cost[i] * pct

# Cash balance
cash_bal = [0.0] * (N + 1)
cash_bal[0] = open_cash
for i in range(N):
    net = cash_in[i] - cash_out[i] - m_interest[i] - m_debt_repay[i]
    cash_bal[i + 1] = cash_bal[i] + net

# AR balance
ar_bal = [0.0] * (N + 1)
ar_bal[0] = open_ar
for i in range(N):
    ar_bal[i + 1] = ar_bal[i] + monthly_revenue[i] - cash_in[i]

# AP balance
ap_bal = [0.0] * (N + 1)
ap_bal[0] = open_ap
for i in range(N):
    ap_bal[i + 1] = ap_bal[i] + monthly_total_cost[i] - cash_out[i]

# Print monthly summary
print(f"{'Month':<8} {'Revenue':>10} {'DC':>10} {'IC':>8} {'EBIT':>10} "
      f"{'Interest':>9} {'Repay':>8} {'CashBal':>10} {'AR':>10} {'AP':>10} {'Debt':>10}")
for i in range(N):
    yr, mo = months[i]
    print(f"{yr}-{mo:02d}  "
          f"{monthly_revenue[i]:>10,.0f} {monthly_dc[i]:>10,.0f} {monthly_ic[i]:>8,.0f} "
          f"{monthly_ebit[i]:>10,.0f} "
          f"{m_interest[i]:>9,.0f} {m_debt_repay[i]:>8,.0f} {cash_bal[i+1]:>10,.0f} "
          f"{ar_bal[i+1]:>10,.0f} {ap_bal[i+1]:>10,.0f} {debt_bal[i+1]:>10,.0f}")

In [ ]:
# ============================================================
# ANSWER QUESTIONS
# ============================================================

def closest(options, value):
    return min(options, key=lambda k: abs(options[k] - value))

# Q1: Sales Revenue in January 2014
idx = month_idx(2014, 1)
val_q1 = monthly_revenue[idx]
q1_opts = {'A': 364000, 'B': 371000, 'C': 349000, 'D': 454000}
q1 = closest(q1_opts, val_q1)
print(f"Q1: Rev Jan 2014 = ${val_q1:,.0f} -> {q1}")

# Q2: Closing Accounts Payable in December 2015
idx = month_idx(2015, 12)
val_q2 = ap_bal[idx + 1]
q2_opts = {'A': 647000, 'B': 764000, 'C': 772000, 'D': 779000}
q2 = closest(q2_opts, val_q2)
print(f"Q2: AP Dec 2015 = ${val_q2:,.0f} -> {q2}")

# Q3: Total Indirect Costs for 3-year forecast
val_q3 = sum(monthly_ic)
q3_opts = {'A': 3900000, 'B': 4000000, 'C': 4200000, 'D': 3600000}
q3 = closest(q3_opts, val_q3)
print(f"Q3: Total IC = ${val_q3:,.0f} -> {q3}")

# Q4: Total interest expense for CY 2014
val_q4 = sum(m_interest[i] for i in range(N) if months[i][0] == 2014)
q4_opts = {'A': 112000, 'B': 125000, 'C': 128000, 'D': 140000}
q4 = closest(q4_opts, val_q4)
print(f"Q4: Interest CY 2014 = ${val_q4:,.0f} -> {q4}")

# Q5: Month when cash runs out
runout = None
for i in range(N):
    if cash_bal[i + 1] < 0:
        runout = months[i]
        break
q5_map = {'A': (2014, 5), 'B': (2014, 6), 'C': (2014, 7), 'D': (2014, 8)}
q5 = 'B'
if runout:
    for k, (y, m) in q5_map.items():
        if runout == (y, m):
            q5 = k
            break
print(f"Q5: Cash runs out = {runout} -> {q5}")

# Q6: Quarter where cash shortfall peaks (lowest quarter-end cash)
q_end_cash = {}
for i in range(N):
    yr, mo = months[i]
    if mo in [3, 6, 9, 12]:
        q_num = (mo - 1) // 3 + 1
        label = f"Q{q_num} {yr}"
        q_end_cash[label] = cash_bal[i + 1]

min_q = min(q_end_cash, key=q_end_cash.get)
q6_map = {'A': 'Q4 2015', 'B': 'Q1 2016', 'C': 'Q3 2015', 'D': 'Q2 2016'}
q6 = 'A'
for k, label in q6_map.items():
    if label == min_q:
        q6 = k
        break
print(f"Q6: Peak shortfall = {min_q} (${q_end_cash[min_q]:,.0f}) -> {q6}")

# Q7: Target debtor balance for 43 debtor days KPI at Jun 30, 2014
# Period: Oct 2013 to Jun 2014 (9 months)
# Debtor days = avg(opening_AR, closing_AR) / revenue * days_in_period
# 43 = avg(600000, target) / rev_9mo * days_9mo
# target = 43 * rev_9mo / days_9mo * 2 - 600000
idx_start = month_idx(2013, 10)
idx_end = month_idx(2014, 6)
rev_9mo = sum(monthly_revenue[idx_start:idx_end + 1])
days_9mo = sum(calendar.monthrange(yr, mo)[1] for yr, mo in months[idx_start:idx_end + 1])
target_ar = 43 * rev_9mo / days_9mo * 2 - open_ar
q7_opts = {'A': 540000, 'B': 560000, 'C': 580000, 'D': 590000}
q7 = closest(q7_opts, target_ar)
print(f"Q7: Target AR = ${target_ar:,.0f} -> {q7}")

# Q8: Interest Coverage Ratio for CY 2015
ebit_2015 = sum(monthly_ebit[i] for i in range(N) if months[i][0] == 2015)
int_2015 = sum(m_interest[i] for i in range(N) if months[i][0] == 2015)
icr_2015 = ebit_2015 / int_2015 if int_2015 > 0 else 0
q8_opts = {'A': 1.2, 'B': 1.3, 'C': 1.4, 'D': 1.5}
q8 = closest(q8_opts, icr_2015)
print(f"Q8: EBIT={ebit_2015:,.0f}, Int={int_2015:,.0f}, ICR={icr_2015:.3f}x -> {q8}")

# Q9: Sensitivity - price +12%, volume -5%, DC per unit maintained
# Applied from Jan 2014 onward
s_rev = []
s_dc = []
s_total_cost = []
for i, (yr, mo) in enumerate(months):
    if yr >= 2014:
        u = monthly_units[i] * 0.95
        r = u * sale_price * 1.12
        d = u * dc_per_unit
    else:
        u = monthly_units[i]
        r = monthly_revenue[i]
        d = monthly_dc[i]
    s_rev.append(r)
    s_dc.append(d)
    s_total_cost.append(d + monthly_ic[i])

# Recompute cash flows for sensitivity
s_cash_in = [0.0] * N
for j, pct in enumerate(cr_open):
    if j < N:
        s_cash_in[j] += open_ar * pct
for i in range(N):
    for j, pct in enumerate(cr_timing):
        t = i + j
        if t < N:
            s_cash_in[t] += s_rev[i] * pct

s_cash_out = [0.0] * N
for j, pct in enumerate(cp_open):
    if j < N:
        s_cash_out[j] += open_ap * pct
for i in range(N):
    for j, pct in enumerate(cp_timing):
        t = i + j
        if t < N:
            s_cash_out[t] += s_total_cost[i] * pct

# Debt schedule stays the same in sensitivity
s_cash_bal = [0.0] * (N + 1)
s_cash_bal[0] = open_cash
for i in range(N):
    net = s_cash_in[i] - s_cash_out[i] - m_interest[i] - m_debt_repay[i]
    s_cash_bal[i + 1] = s_cash_bal[i] + net

idx_dec2015 = month_idx(2015, 12)
val_q9 = s_cash_bal[idx_dec2015 + 1]
q9_opts = {'A': 185000, 'B': 260000, 'C': -60000, 'D': 360000}
q9 = closest(q9_opts, val_q9)
print(f"Q9: Sensitivity cash end CY2015 = ${val_q9:,.0f} -> {q9}")

print("\nAll answers:", [q1, q2, q3, q4, q5, q6, q7, q8, q9])

In [ ]:
answers = {
    "question1": q1,
    "question2": q2,
    "question3": q3,
    "question4": q4,
    "question5": q5,
    "question6": q6,
    "question7": q7,
    "question8": q8,
    "question9": q9,
}
print("answers =", answers)